# Pipeline 06: Synthetic Data Distillation

**Goal:** Generate large-scale (prompt, JSON) training pairs by commanding Claude to produce diverse permutations from seed examples. Builds the dataset for Qwen fine-tuning.

**Depends on:** Pipeline 05 (may already have some manually-verified records)

**Runtime:** CPU fine (API calls only).

In [ ]:
# Setup
from google.colab import drive
drive.mount('/content/drive')

!pip install anthropic tqdm -q

import os, re, json, random, time
import shutil
from tqdm.notebook import tqdm
from google.colab import userdata
from anthropic import Anthropic

PROJECT = '/content/drive/MyDrive/photo-style-rl'
DATASET_PATH = f'{PROJECT}/data/slm_training_dataset.jsonl'
os.makedirs(os.path.dirname(DATASET_PATH), exist_ok=True)

# Schema constants + validate_payload come from shared.py — single source of truth
shutil.copy(f'{PROJECT}/src/shared.py', '/content/shared.py')
from shared import (
    AVAILABLE_REGIONS,
    VALID_TONE_CURVE_KEYS, VALID_COLOR_NAMES, VALID_HSL_KEYS, VALID_BASIC_KEYS,
    validate_payload,
)

client = Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

seed_prompts = [
    "brighten the subjects and make their skin tones warmer",
    "make the sunset more dramatic and lift the shadows so it's not too contrasty",
    "make this feel more like a vintage film aesthetic of an old Asian street",
    "reduce the highlights and raise the shadows while overall brightening up the subject's face",
    "add some noise and give this a cinematic movie haze",
    "i want this photo to look as though i took it on a fujifilm x100vi",
    "change the orange to be more faded and make the sky more teal",
    "make this in the style of director Wong Kar-Wai's film aesthetic",
    "change the vibe to a late night cyberpunk neon street scene",
    "reduce the contrast on the person's face so the impurities don't stand out",
    "the top left corner is too bright — reduce it slightly and shift the green hue to olive",
    "selectively apply radial brushes around the subject and darken everything else",
    "you are an expert photographer — edit this in what you think would look best",
    "i want the sunset to be more vibrant and a bit more pink to represent dusk",
    "can you make this feel more punchy and powerful while still tasteful?",
    "mimic a real-life documentary photo worthy of a Pulitzer",
]

print(f"Distillation engine ready. {len(seed_prompts)} seed prompts.")
print(f"Schema: {len(VALID_BASIC_KEYS)} basic keys, {len(VALID_COLOR_NAMES)} color names")
existing = sum(1 for _ in open(DATASET_PATH)) if os.path.exists(DATASET_PATH) else 0
print(f"Existing records: {existing}")


In [ ]:
# Generation + validation
# validate_payload is imported from shared.py — single definition, used here and in NB07.

def generate_batch(batch_size=10):
    """Generate synthetic (prompt, JSON) pairs from Claude."""
    sys_prompt = f"""You are an expert photo editor generating synthetic training data.
Generate {batch_size} NEW, diverse photo editing prompts inspired by the seeds.
Vary tone (casual, technical, abstract) and requested edits.

For each prompt, provide correct JSON math for our rendering engine.
Available regions: {AVAILABLE_REGIONS}

STRICT SCHEMA — only these keys are allowed:
- Top level: region names from the available list
- Per region: "tone_curve" and/or "color_mixer" and/or basic keys
- Basic keys: exposure [-5,5], contrast, highlights, shadows, whites, blacks, temperature, tint, clarity, vibrance, saturation, texture, dehaze [-100,100]
- tone_curve keys: {list(VALID_TONE_CURVE_KEYS)}, values: -100 to 100
- color_mixer keys: {list(VALID_COLOR_NAMES)}, each with {{h: -180 to 180, s: -100 to 100, l: -100 to 100}}

Output ONLY a valid JSON array:
[{{"user_prompt": "...", "json_payload": {{"global": {{...}}, "sky": {{...}}}}}}]"""

    seeds_text = "\n".join([f"- {p}" for p in random.sample(seed_prompts, min(8, len(seed_prompts)))])

    try:
        response = client.messages.create(
            model="claude-sonnet-4-20250514", max_tokens=3000,
            system=sys_prompt,
            messages=[{"role": "user", "content": f"Seeds:\n{seeds_text}\n\nGenerate {batch_size} new pairs now."}],
        )
        text = response.content[0].text
        match = re.search(r'\[[\s\S]*\]', text)
        if match:
            return json.loads(match.group(0))
    except Exception as e:
        print(f"  API error: {e}")
        if "rate" in str(e).lower() or "429" in str(e):
            time.sleep(30)
    return []


# Run generation
TOTAL_BATCHES = 20  # 200 target pairs
successful = 0
rejected = 0

print(f"Generating ~{TOTAL_BATCHES * 10} synthetic pairs...\n")

with open(DATASET_PATH, 'a') as f:
    for i in tqdm(range(TOTAL_BATCHES), desc="Distilling"):
        batch = generate_batch(batch_size=10)
        for item in batch:
            payload = item.get("json_payload", {})
            if validate_payload(payload):
                record = {
                    "messages": [
                        {"role": "user", "content": item.get("user_prompt", "")},
                        {"role": "assistant", "content": json.dumps(payload)},
                    ]
                }
                f.write(json.dumps(record) + "\n")
                successful += 1
            else:
                rejected += 1

print(f"\nDone. Added {successful}, rejected {rejected} (bad schema).")
total = sum(1 for _ in open(DATASET_PATH))
print(f"Total dataset: {total} records")

# Sample verification
if total > 0:
    with open(DATASET_PATH) as f:
        sample = json.loads(random.choice(f.readlines()))
    print(f"\nSample prompt: {sample['messages'][0]['content']}")
    print(f"Sample output: {json.dumps(json.loads(sample['messages'][1]['content']), indent=2)}")
